# ShopFlow Experimentation Analysis

## Purpose and scope

This notebook brings the tested experimentation helpers together in one reproducible analysis of ShopFlow's **product-view-to-add-to-cart conversion** outcome.

- **Unit of analysis:** user.
- **Eligibility:** a user with a consistent experiment assignment and at least one valid `view_product` event.
- **Anchor:** the user's earliest qualifying product view.
- **Conversion:** at least one `add_to_cart` event in the same session, at or after that view and within the inclusive seven-day window.
- **Comparison:** treatment versus control, with a two-sided pooled two-proportion z-test and an unpooled Wald confidence interval.
- **Interpretation:** exploratory analysis of synthetic data. Recorded group labels and observed differences do not prove randomization, exposure delivery, or a causal effect.

This is a separate outcome from purchase conversion discussed in earlier notebooks. The pre-analysis plan and results documentation remain the reference for unresolved design and launch decisions.

**Important design limitations:** the event data has no explicit assignment timestamp, so this analysis cannot verify that the anchor view occurred after assignment. Because eligibility requires a product view, and treatment may affect viewing, the comparison can be subject to selection bias. It is a view-conditioned exploratory comparison, not an intention-to-treat estimate or a verified causal treatment effect.


## 1. Reproducible setup

Run from the repository root. The helper functions live in `src/`; this notebook imports them instead of reimplementing the calculations.


In [ ]:
from pathlib import Path
import sys
import math

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    # Also support launching Jupyter from inside notebooks/.
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks/.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experimentation import (
    achieved_power,
    difference_confidence_interval,
    required_sample_size_per_group,
    two_proportion_z_test,
)
from src.experiment_integrity import validate_experiment_integrity
from src.experiment_outcomes import summarize_user_level_cart_conversion

DATA_PATH = ROOT / "data" / "processed" / "clean_events.csv"
PREANALYSIS_PATH = ROOT / "docs" / "EXPERIMENT_PREANALYSIS_PLAN.md"
RESULTS_PATH = ROOT / "docs" / "EXPERIMENTATION_RESULTS.md"

assert DATA_PATH.exists(), f"Missing processed events: {DATA_PATH}"
print("Repository:", ROOT)
print("Input:", DATA_PATH)
print("Pre-analysis plan exists:", PREANALYSIS_PATH.exists())
print("Results documentation exists:", RESULTS_PATH.exists())


## 2. Load and validate the processed events

The processed CSV is the primary source for this notebook. No database writes are performed. The database is not needed for the user-level cart outcome, so it is deliberately not opened here.


In [ ]:
events = pd.read_csv(DATA_PATH)

required_columns = {
    "user_id", "timestamp", "session_id", "event_name", "experiment_group"
}
missing_columns = required_columns - set(events.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce", utc=True)

print(f"Rows: {len(events):,}")
print(f"Unique users (raw event rows): {events['user_id'].nunique():,}")
print("\nEvent counts:")
display(events["event_name"].value_counts(dropna=False).rename_axis("event").to_frame("events"))
print("\nRecorded group labels by event row (not assignment denominators):")
display(events["experiment_group"].value_counts(dropna=False).rename_axis("group").to_frame("event_rows"))

required_event_fields = ["user_id", "timestamp", "session_id", "event_name", "experiment_group"]
print("\nMissing values in required fields after timestamp parsing:")
display(events[required_event_fields].isna().sum().rename("missing_values").to_frame())

assert events["timestamp"].notna().all(), "Invalid timestamps need investigation."
assert events["user_id"].notna().all(), "Missing user IDs need investigation."
assert events["event_name"].notna().all(), "Missing event names need investigation."
assert set(events["experiment_group"].dropna().unique()).issubset({"control", "treatment"}), (
    "Unexpected experiment-group labels found; inspect before analysis."
)


## 3. Assignment integrity and exposure diagnostics

These diagnostics describe recorded data. A sample-ratio mismatch (SRM) check is only meaningful when the expected allocation is known and the observations are suitable assignment units. Event rows are not independent assignment units, so inspect the helper's reported user-level assignment diagnostics rather than interpreting event-row counts as an SRM test.

The integrity helper defaults to `purchase` as its outcome diagnostic; the primary outcome in this notebook is separately constructed below.


In [ ]:
integrity = validate_experiment_integrity(events)
print("Experiment integrity diagnostics:")
display(pd.json_normalize(integrity).T.rename(columns={0: "value"}))

print("\nUnique users by recorded group label:")
user_group_pairs = events[["user_id", "experiment_group"]].dropna().drop_duplicates()
display(
    user_group_pairs.groupby("experiment_group")["user_id"]
    .nunique().rename("unique_users").to_frame()
)

assignment_counts = user_group_pairs.groupby("user_id")["experiment_group"].nunique()
conflicting_users = int((assignment_counts > 1).sum())
print(f"Users recorded in multiple groups: {conflicting_users:,}")
print(
    "Caveat: recorded consistency cannot establish that assignment was randomized "
    "or that every assigned user received the intended experience."
)


## 4. Construct the primary user-level cart-conversion outcome

Use the shared, tested outcome helper with a fixed seven-day inclusive attribution window. Users without a qualifying product view are reported separately rather than treated as nonconverters in the eligible-user conversion rate. Users with conflicting assignments are excluded by the helper and reported.

The dataset has no explicit assignment timestamp, so the helper cannot verify that the first recorded view followed assignment. Since eligibility is conditioned on viewing a product, treatment-related changes in viewing can create selection bias. Interpret this as a descriptive, view-conditioned metric, not an intention-to-treat or causal estimate.


In [ ]:
outcome = summarize_user_level_cart_conversion(events, window_days=7)
print("Outcome helper summary:")
display(pd.json_normalize(outcome).T.rename(columns={0: "value"}))


group_order = ["control", "treatment"]
group_results = outcome["by_group"]

outcome_rows = []
for group in group_order:
    result = group_results[group]
    outcome_rows.append({
        "experiment_group": group,
        "assigned_users": result["assigned_users"],
        "eligible_users": result["eligible_users"],
        "converted_users": result["converted_users"],
        "conversion_rate": result["conversion_rate"],
        "users_without_view": result["users_without_view"],
    })

outcome_table = pd.DataFrame(outcome_rows)
display(outcome_table)


## 5. Compare treatment and control

The helper output is authoritative for the attribution rule and eligibility. This section extracts the user-level eligible and converted counts for the statistical comparison. If the helper return schema changes, stop and inspect the returned summary rather than guessing denominators.


In [ ]:
control_n = int(group_results["control"]["eligible_users"])
treatment_n = int(group_results["treatment"]["eligible_users"])

control_x = int(group_results["control"]["converted_users"])
treatment_x = int(group_results["treatment"]["converted_users"])


assert control_n > 0 and treatment_n > 0

pooled_rate = (control_x + treatment_x) / (control_n + treatment_n)

expected_counts = {
    "control_converters": control_n * pooled_rate,
    "control_nonconverters": control_n * (1 - pooled_rate),
    "treatment_converters": treatment_n * pooled_rate,
    "treatment_nonconverters": treatment_n * (1 - pooled_rate),
}

print("Expected counts under the pooled null:")
display(pd.Series(expected_counts, name="expected_count").to_frame())

if min(expected_counts.values()) < 5:
    print(
        "WARNING: At least one expected cell count is below 5. "
        "The normal approximation for the z-test may be unreliable; "
        "consider an appropriate alternative before interpreting results."
    )


test = two_proportion_z_test(control_x, control_n, treatment_x, treatment_n)
ci_low, ci_high = difference_confidence_interval(
    control_x, control_n, treatment_x, treatment_n, confidence_level=0.95
)

comparison = pd.DataFrame([
    {
        "group": "control",
        "converters": control_x,
        "eligible_users": control_n,
        "conversion_rate": control_x / control_n,
    },
    {
        "group": "treatment",
        "converters": treatment_x,
        "eligible_users": treatment_n,
        "conversion_rate": treatment_x / treatment_n,
    },
])
display(comparison.style.format({"conversion_rate": "{:.2%}"}))

print(f"Absolute difference: {test['absolute_lift']:.4%} ({test['absolute_lift'] * 100:.2f} percentage points)")
relative_lift = test["relative_lift"]
if pd.notna(relative_lift):
    print(f"Relative difference: {relative_lift:.2%}")
else:
    print("Relative difference: N/A (control conversion rate is zero)")
print(f"95% unpooled Wald CI: [{ci_low:.4%}, {ci_high:.4%}]")
print(f"Two-sided pooled z-test p-value: {test['p_value']:.6g}")


In [ ]:
ax = comparison.set_index("group")["conversion_rate"].reindex(group_order).plot(
    kind="bar", rot=0, ylim=(0, 1), figsize=(7, 4),
    title="Eligible-user product-view-to-cart conversion"
)
ax.set_ylabel("Conversion rate")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.show()


### Interpretation guardrails

- The p-value measures compatibility with a null hypothesis under the test's assumptions; it does not measure the probability that the null is true.
- The Wald confidence interval is an approximation and may perform poorly with small counts or extreme rates.
- Statistical significance does not establish business value.
- The outcome is based on synthetic events. Do not present this as a production experiment or a causal finding.
- Do not change the attribution window or primary outcome after looking at the results without documenting the change.


## 6. Sample-size and power planning

Use the observed eligible control rate as a provisional planning baseline. The candidate absolute minimum detectable effects below are illustrative scenarios, **not approved business thresholds**. Planning assumes equal-sized independent groups, a fixed horizon, two-sided alpha of 0.05, and 80% target power.


In [ ]:
baseline = control_x / control_n
alpha = 0.05
target_power = 0.80
candidate_mdes = [0.01, 0.02, 0.05]

planning_rows = []
for mde in candidate_mdes:
    if baseline + mde >= 1:
        continue
    required_n = required_sample_size_per_group(
        baseline_rate=baseline,
        minimum_detectable_effect=mde,
        alpha=alpha,
        power=target_power,
    )
    current_n = min(control_n, treatment_n)
    planning_rows.append({
        "baseline_rate": baseline,
        "illustrative_absolute_MDE": mde,
        "required_n_per_group": required_n,
        "required_total_n": 2 * required_n,
        "current_smaller_eligible_group_n": current_n,
        "approx_power_at_current_n": achieved_power(
            baseline_rate=baseline,
            minimum_detectable_effect=mde,
            n_per_group=current_n,
            alpha=alpha,
        ),
    })

planning = pd.DataFrame(planning_rows)
display(planning.style.format({
    "baseline_rate": "{:.2%}",
    "illustrative_absolute_MDE": "{:.1%}",
    "approx_power_at_current_n": "{:.1%}",
}))


### Planning decisions still needed

Before a real experiment could be approved, stakeholders would need to set the smallest worthwhile absolute lift and guardrails, confirm the assignment unit and allocation, validate exposure delivery and outcome completeness, define a fixed stopping rule, and assess attrition and instrumentation. These synthetic-data calculations cannot resolve those questions.


## 7. Sensitivity, assumptions, and conclusions

Review whether the fixed seven-day same-session rule is appropriate for the product journey. A different window, session rule, eligibility definition, or assignment policy can change the denominator and observed conversion rate. Such alternatives should be pre-specified or clearly labeled as sensitivity analyses, not substituted silently for the primary result.

### Reproducibility checks

The following checks validate the statistical interface and internal consistency without pinning the notebook to a particular p-value or requiring synthetic outcomes to demonstrate a desired effect.


In [ ]:
assert control_n > 0 and treatment_n > 0
assert 0 <= control_x <= control_n
assert 0 <= treatment_x <= treatment_n
assert 0 <= test["p_value"] <= 1
assert ci_low <= ci_high
assert all(row["required_n_per_group"] > 0 for row in planning_rows)

print("Reproducibility checks passed.")
print(f"Eligible control users: {control_n:,}; converters: {control_x:,}")
print(f"Eligible treatment users: {treatment_n:,}; converters: {treatment_x:,}")
print(f"Outcome window: 7 days, same session, inclusive endpoint")
print("Causal effect established:", outcome.get("causal_effect_established", False))
print("Interpretation: exploratory synthetic-data analysis only.")
print("\nReference documents:")
print("-", PREANALYSIS_PATH.relative_to(ROOT))
print("-", RESULTS_PATH.relative_to(ROOT))


## Final reporting checklist

- [ ] Confirm the pre-analysis plan and business-approved MDE before any production experiment.
- [ ] Validate randomization, assignment consistency, exposure delivery, and data completeness.
- [ ] Confirm the primary metric, attribution window, stopping rule, and guardrails in advance.
- [ ] Report eligible denominators, converters, absolute lift, relative lift, confidence interval, and p-value together.
- [ ] Assess practical significance and guardrail outcomes, not p-values alone.
- [ ] Label this notebook's results as synthetic and exploratory.
